# GPU 训练性能分析

本笔记整理了 LLM 训练中关于显存占用、计算力评估的核心概念，
以及主流 GPU 硬件的性能参数对比，帮助在选卡和调参时做出合理决策。

## 1. 显存占用分析

训练时显存由四部分构成：

```
总显存 = 模型参数 + 梯度 + 优化器状态 + 激活值
```

### 1.1 模型参数

| 精度 | 每个参数占用 | 1B 参数模型 |
|------|------------|------------|
| FP32 | 4 bytes | 4 GB |
| FP16 / BF16 | 2 bytes | 2 GB |
| INT8 | 1 byte | 1 GB |
| INT4 | 0.5 bytes | 0.5 GB |

### 1.2 梯度

梯度与参数等大，精度通常与参数一致：

```
梯度显存 = 参数量 × 每参数字节数
```

### 1.3 优化器状态（AdamW）

AdamW 需要存储一阶矩（m）和二阶矩（v），各与参数等大，且通常用 FP32 保存精度：

```
AdamW 优化器状态 = 参数量 × 4 bytes × 2 = 参数量 × 8 bytes
```

### 1.4 激活值（Activation）

激活值与 batch size、序列长度正相关，是显存占用中**最灵活**的部分：

```
激活值 ∝ batch_size × seq_len × hidden_size × num_layers
```

### 1.5 混合精度训练的完整显存公式

混合精度（AMP）下，模型同时保存 FP16 和 FP32 副本：

```
总显存 = 参数(FP16) + 参数(FP32副本) + 梯度(FP16) + AdamW状态(FP32) + 激活值
       = P×2 + P×4 + P×2 + P×8 + 激活值
       = P×16 bytes + 激活值
```

其中 P 为参数量（个数）。

**经验估算（混合精度 + AdamW）：**

| 模型规模 | 最低显存估算 |
|---------|------------|
| 124M (GPT-2) | ~2 GB（不含激活）|
| 7B (LLaMA-2) | ~112 GB |
| 13B | ~208 GB |
| 70B | ~1.1 TB |

In [ ]:
def estimate_training_memory(params_billion, precision='fp16', optimizer='adamw'):
    """
    估算训练所需显存（不含激活值）
    
    params_billion: 模型参数量（单位：十亿）
    precision: 'fp32', 'fp16', 'bf16'
    optimizer: 'adamw', 'sgd'
    """
    P = params_billion * 1e9

    if precision in ('fp16', 'bf16'):
        # 混合精度：参数FP16 + FP32副本 + 梯度FP16 + AdamW状态FP32
        model_bytes   = P * 2          # 参数 FP16
        fp32_copy     = P * 4          # FP32 主权重副本（AMP 必须）
        grad_bytes    = P * 2          # 梯度 FP16
        optim_bytes   = P * 8 if optimizer == 'adamw' else P * 4  # m + v (FP32)
    else:  # fp32
        model_bytes   = P * 4
        fp32_copy     = 0
        grad_bytes    = P * 4
        optim_bytes   = P * 8 if optimizer == 'adamw' else P * 4

    total_bytes = model_bytes + fp32_copy + grad_bytes + optim_bytes
    total_gb = total_bytes / 1024**3

    print(f"模型参数量：{params_billion}B")
    print(f"精度：{precision}，优化器：{optimizer}")
    print(f"  模型参数：  {model_bytes/1024**3:.1f} GB")
    if fp32_copy:
        print(f"  FP32副本：  {fp32_copy/1024**3:.1f} GB")
    print(f"  梯度：      {grad_bytes/1024**3:.1f} GB")
    print(f"  优化器状态：{optim_bytes/1024**3:.1f} GB")
    print(f"  合计（不含激活）：{total_gb:.1f} GB")
    return total_gb

# GPT-2
estimate_training_memory(0.124)
print()
# LLaMA-2 7B
estimate_training_memory(7)
print()
# LLaMA-2 70B
estimate_training_memory(70)

## 2. 计算量（FLOPs）分析

### 2.1 Transformer 单次前向传播 FLOPs

对于标准 Transformer（仅 attention + FFN），单次前向传播的近似公式：

```
前向 FLOPs ≈ 2 × P × T
```

其中：
- `P`：模型参数量
- `T`：输入 token 数（batch_size × seq_len）

**为什么是 2？** 每个参数参与一次乘法和一次加法（MAC = Multiply-Accumulate）。

### 2.2 训练总 FLOPs

训练 = 前向 + 反向，反向约为前向的 2 倍：

```
训练 FLOPs ≈ 6 × P × T_total
```

其中 `T_total = 训练总 token 数 = num_samples × seq_len`

### 2.3 理论训练时间估算

```
理论时间 = 训练总FLOPs / (GPU算力 × GPU利用率 × GPU数量)
```

实际 GPU 利用率通常在 30%-60%（受 IO、通信等影响）。

In [ ]:
def estimate_training_time(
    params_billion,
    total_tokens_billion,
    gpu_tflops,
    num_gpus=1,
    gpu_utilization=0.45
):
    """
    估算训练时间

    params_billion:        模型参数量（B）
    total_tokens_billion:  训练 token 总量（B）
    gpu_tflops:            GPU 峰值算力（TFLOPS，BF16/FP16）
    num_gpus:              GPU 数量
    gpu_utilization:       实际利用率（通常 0.3~0.6）
    """
    total_flops = 6 * params_billion * 1e9 * total_tokens_billion * 1e9
    effective_flops_per_sec = gpu_tflops * 1e12 * num_gpus * gpu_utilization
    seconds = total_flops / effective_flops_per_sec
    hours = seconds / 3600

    print(f"模型：{params_billion}B 参数，训练 {total_tokens_billion}B tokens")
    print(f"硬件：{num_gpus}x GPU @ {gpu_tflops} TFLOPS，利用率 {gpu_utilization:.0%}")
    print(f"训练总 FLOPs：{total_flops:.2e}")
    print(f"预计时间：{hours:.1f} 小时 ({hours/24:.1f} 天)")
    return hours

# CodeParrot 实验（本课程任务）
# GPT-2 124M，训练集约 16.7M 条 × 128 tokens ≈ 2.1B tokens
print("=== 本课程任务：GPT-2 on CodeParrot ===")
estimate_training_time(0.124, 2.1, gpu_tflops=835, num_gpus=2, gpu_utilization=0.85)  # 2x H100 NVL
print()
estimate_training_time(0.124, 2.1, gpu_tflops=312, num_gpus=1, gpu_utilization=0.45)  # 1x A100 PCIe
print()

# LLaMA-3 8B 预训练规模（参考）
print("=== 参考：LLaMA-3 8B 级别预训练 (15T tokens) ===")
estimate_training_time(8, 15000, gpu_tflops=835, num_gpus=2048, gpu_utilization=0.45)

## 3. 有效 Batch Size 与梯度累积

```
有效 batch size = per_device_batch × num_gpus × gradient_accumulation_steps
```

### 为什么有效 batch size 重要？

- 太小（< 32）：梯度噪声大，收敛不稳定
- 太大（> 4096）：泛化性下降（sharp minima 问题）
- 常用范围：**256 ~ 2048**

### 梯度累积的代价

梯度累积步数越多，参数更新频率越低，等效于减少了每 epoch 的更新次数，
能用真实 batch（更多 GPU 或更大显存）替代时，应优先选择真实 batch。

| 配置 | 有效 batch | 每步真实计算量 | 参数更新频率 |
|------|-----------|--------------|------------|
| batch=32, accum=8, 1 GPU | 256 | 低 | 低 |
| batch=128, accum=2, 1 GPU | 256 | 高 | 高（更快）|
| batch=128, accum=1, 2 GPU | 256 | 高 | 高（更快）|

In [ ]:
def batch_config_analysis(total_samples, per_device_batch, num_gpus, grad_accum):
    effective_batch = per_device_batch * num_gpus * grad_accum
    total_steps = total_samples // effective_batch
    update_steps = total_steps  # 每 step 都更新（grad_accum 已折叠进 effective_batch）

    print(f"per_device_batch={per_device_batch}, num_gpus={num_gpus}, grad_accum={grad_accum}")
    print(f"  有效 batch size：{effective_batch}")
    print(f"  总训练步数：{total_steps:,}")
    print()

total_samples = 16_702_061  # CodeParrot 训练集

print("=== CodeParrot 训练集不同配置对比 ===")
batch_config_analysis(total_samples, per_device_batch=32,  num_gpus=1, grad_accum=8)
batch_config_analysis(total_samples, per_device_batch=128, num_gpus=1, grad_accum=2)
batch_config_analysis(total_samples, per_device_batch=128, num_gpus=2, grad_accum=2)
batch_config_analysis(total_samples, per_device_batch=512, num_gpus=2, grad_accum=1)

## 4. 主流 GPU 性能参数对比

### 4.1 数据中心 GPU

| GPU | 显存 | 显存带宽 | BF16算力 | 互联方式 | TDP |
|-----|------|---------|---------|---------|-----|
| H100 SXM 80G | 80GB HBM3 | 3.35 TB/s | 1,979 TFLOPS | NVLink 4.0 | 700W |
| H100 NVL 94G | 94GB HBM3e | 3.9 TB/s | 835 TFLOPS\* | NVLink Bridge | 400W |
| H100 PCIe 80G | 80GB HBM3 | 2.0 TB/s | 1,513 TFLOPS | PCIe 5.0 | 350W |
| A100 SXM 80G | 80GB HBM2e | 2.0 TB/s | 312 TFLOPS | NVLink 3.0 | 400W |
| A100 PCIe 80G | 80GB HBM2e | 1.94 TB/s | 312 TFLOPS | PCIe 4.0 | 300W |
| A100 PCIe 40G | 40GB HBM2e | 1.55 TB/s | 312 TFLOPS | PCIe 4.0 | 250W |
| A10G | 24GB GDDR6 | 600 GB/s | 31.2 TFLOPS | PCIe 4.0 | 150W |

> \* H100 NVL 的 835 TFLOPS 为 sparsity 关闭状态下的 dense 算力；开启 sparsity 可达 1,671 TFLOPS。

### 4.2 消费级 GPU

| GPU | 显存 | 显存带宽 | FP16算力 | 互联 | TDP |
|-----|------|---------|---------|------|-----|
| RTX 4090 | 24GB GDDR6X | 1.0 TB/s | 165 TFLOPS | PCIe 4.0 | 450W |
| RTX 4080 | 16GB GDDR6X | 717 GB/s | 97 TFLOPS | PCIe 4.0 | 320W |
| RTX 3090 Ti | 24GB GDDR6X | 1.0 TB/s | 80 TFLOPS | PCIe 4.0 | 450W |
| RTX 3090 | 24GB GDDR6X | 936 GB/s | 71 TFLOPS | PCIe 4.0 | 350W |

### 4.3 综合性价比（训练场景）

| 场景 | 推荐 GPU | 原因 |
|------|---------|------|
| 课程练习 / 小模型（<1B） | RTX 4090 / A100 PCIe 40G | 够用，成本低 |
| 中型模型微调（1B~13B） | A100 PCIe 80G | 显存充足，性价比高 |
| 大模型微调（13B~70B）| 2~4x A100 SXM / H100 SXM | NVLink 多卡扩展好 |
| 超大模型预训练（70B+）| 8x H100 SXM 集群 | 最高算力 + NVSwitch |

In [ ]:
# 主流 GPU 参数对比可视化
import matplotlib.pyplot as plt
import numpy as np

gpus = {
    'H100 SXM 80G':   {'bf16_tflops': 1979, 'vram_gb': 80,  'color': '#e63946'},
    'H100 NVL 94G':   {'bf16_tflops': 835,  'vram_gb': 94,  'color': '#e63946'},
    'H100 PCIe 80G':  {'bf16_tflops': 1513, 'vram_gb': 80,  'color': '#e63946'},
    'A100 SXM 80G':   {'bf16_tflops': 312,  'vram_gb': 80,  'color': '#457b9d'},
    'A100 PCIe 80G':  {'bf16_tflops': 312,  'vram_gb': 80,  'color': '#457b9d'},
    'A100 PCIe 40G':  {'bf16_tflops': 312,  'vram_gb': 40,  'color': '#457b9d'},
    'RTX 4090':       {'bf16_tflops': 165,  'vram_gb': 24,  'color': '#2a9d8f'},
    'RTX 3090':       {'bf16_tflops': 71,   'vram_gb': 24,  'color': '#2a9d8f'},
}

names = list(gpus.keys())
tflops = [gpus[g]['bf16_tflops'] for g in names]
vram = [gpus[g]['vram_gb'] for g in names]
colors = [gpus[g]['color'] for g in names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# BF16 算力对比
bars1 = ax1.barh(names, tflops, color=colors)
ax1.set_xlabel('BF16 算力 (TFLOPS)')
ax1.set_title('BF16 计算算力对比')
for bar, val in zip(bars1, tflops):
    ax1.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
             f'{val}', va='center', fontsize=9)

# 显存对比
bars2 = ax2.barh(names, vram, color=colors)
ax2.set_xlabel('显存 (GB)')
ax2.set_title('显存容量对比')
for bar, val in zip(bars2, vram):
    ax2.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f'{val} GB', va='center', fontsize=9)

# 图例
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#e63946', label='H100 系列'),
    Patch(facecolor='#457b9d', label='A100 系列'),
    Patch(facecolor='#2a9d8f', label='消费级 RTX'),
]
ax1.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('gpu_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 多卡训练：互联带宽的影响

多卡训练时，每次参数更新需要通过 **All-Reduce** 同步所有 GPU 的梯度：

```
All-Reduce 通信量 = 2 × (N-1)/N × 参数量 × 字节数
                  ≈ 2 × 参数量 × 字节数  （N 较大时）
```

### 互联方式对比

| 互联方式 | 带宽 | 典型卡型 | 多卡扩展效率 |
|---------|------|---------|------------|
| NVSwitch（DGX）| ~900 GB/s（全互联）| H100 SXM, A100 SXM | 95%+ |
| NVLink Bridge | ~900 GB/s（点对点）| H100 NVL（成对）| 90%+（2卡）|
| PCIe 5.0 | ~128 GB/s | H100 PCIe | 60-80% |
| PCIe 4.0 | ~64 GB/s | RTX 4090, A100 PCIe | 40-70% |

### 实际影响

GPT-2（124M）梯度大小：
```
124M × 2 bytes (FP16) = 248 MB
```

| 互联方式 | All-Reduce 耗时（248MB）| 占比（假设每步10ms计算）|
|---------|----------------------|----------------------|
| NVLink | ~0.3 ms | ~3%（可忽略）|
| PCIe 4.0 | ~4 ms | ~40%（显著瓶颈）|

> **结论**：小模型多卡用 PCIe 互联时，通信开销占比大，扩展效率低。
> 大模型（7B+）每步计算时间长，通信占比相对小，PCIe 多卡也可接受。

## 6. 本课程任务（CodeParrot GPT-2）实测分析

### 任务配置
- 模型：GPT-2（124M 参数）
- 数据集：codeparrot-ds，16.7M 样本 × 128 tokens ≈ **2.1B tokens**
- 硬件：2x H100 NVL 94G

### 实测数据

| 配置 | 显存占用 | 速度 | 总步数 | 预计时间 |
|------|---------|------|-------|--------|
| batch=128, accum=2 | 17%（16GB）| 6.63 it/s | 32,622 | ~1h21min |
| batch=512, accum=1 | 60%（56GB）| 3.39 it/s | 16,311 | ~1h19min |

### 为什么增大 batch 没有加速？

```
样本吞吐量（samples/s）：
  batch=128：6.63 × 128 × 2 = 1,697 samples/s
  batch=512：3.39 × 512 × 2 = 3,472 samples/s  ← 翻倍！

但总步数也减半（32622 → 16311），时间保持不变。
```

**根本原因**：GPU 计算利用率已达 **95-99%**，是计算瓶颈而非显存瓶颈。
增大 batch 提高了每步工作量，但无法让 GPU 跑得更快。

### 硬件选型建议

| 硬件 | 预估时间 | 性价比 |
|------|---------|-------|
| 2x H100 NVL | ~1 小时 | 低（杀鸡用牛刀）|
| 1x A100 PCIe 80G | ~3-5 小时 | 中 |
| 1x RTX 4090 | ~4-6 小时 | 高（最适合练习）|

## 7. 常用优化手段汇总

| 优化手段 | 效果 | 显存变化 | 适用场景 |
|---------|------|---------|--------|
| **混合精度 (bf16/fp16)** | 速度 +30-50% | -50% | 几乎所有场景 |
| **torch.compile** | 速度 +20-30% | 不变 | PyTorch 2.0+，需预热 |
| **Flash Attention 2** | 速度 +2-4x | -30-50% | 序列长度 ≥ 1K |
| **Flash Attention 3** | 速度 +1.5-2x（vs FA2）| 同FA2 | H100，CUDA 12.6+，序列长 |
| **Gradient Checkpointing** | 速度 -20%（重计算）| -60% | 显存不足时 |
| **Fused AdamW** | 速度 +5-10% | 不变 | 优化器步骤加速 |
| **增大 batch size** | 吞吐量提升 | 增加 | 显存有余量时 |
| **NVLink 多卡** | 近线性扩展 | 按卡数增加 | 模型需要或追求速度 |

### 优先级建议

```
1. bf16/fp16        ← 最简单，收益最大，必开
2. 增大 batch       ← 有余量就增大，减少通信开销
3. torch.compile    ← 免费加速，首次有预热开销
4. Fused AdamW      ← 一行代码，小幅提升
5. Flash Attention  ← 长序列必备，短序列意义不大
6. 多卡             ← 单卡不够用时，优先 NVLink 互联
7. Gradient Ckpt    ← 显存实在不够时的最后手段
```